# Lecture 03 · Practical Quant Trading Lab
## From Trading Paradigms to a Tested Strategy — on Real Market Data

**Course:** Algorithmic Trading with Python · **Companion to:** *Lecture 03 – Trading Strategies & Paradigms* (Ali Habibnia)
**Builds on:** Lecture 00 (Python), Lecture 01 (Data Handling), Lecture 02 (Stock Screener)
**Not required:** Lecture 04–08 material. Anything from there appears only as a labelled optional extension.

> ⚠️ **Read this first — what this lab is and is not**
> - This is a **research workbook, not a tutorial**. Core cells contain `# TODO` placeholders. You make the implementation decisions; passing the `CHECK` cells is how you know a decision is internally consistent.
> - **No complete solutions are provided in this notebook.** Short reference patterns are given (§Appendix A); assembling them into strategies is the work.
> - **All market data in this lab is real historical data** downloaded from Yahoo Finance via `yfinance`. There is no synthetic, seeded, or fabricated price series anywhere in this lab, and no performance number is printed in advance. Every result you report must be produced by *your* code running on *your* download.
> - Nothing here is investment advice, and a profitable historical backtest is **not** evidence of future profit.

---

### How to work through it

| Tag | Meaning |
|---|---|
| `TODO` | You write the code. Hints name the technique, never the full line. |
| `CHECK` | Runnable `assert` cell. Run it immediately. A failure tells you which invariant broke — read it, don't delete it. |
| `REFLECT` | Short written answer, in your own words, in the cell provided. Graded. |
| `STRETCH` | Optional. Never a prerequisite for a later required module. |

**Working rules:** run cells top-to-bottom · never edit a `CHECK` cell to make it pass · keep one notebook-wide naming convention · if Yahoo Finance is unreachable, stop and fix the download (§M0.4) — **do not** substitute invented data.

**Time budget:** ≈ 8–12 hours. **Deliverable:** this notebook, run end-to-end (`Kernel → Restart & Run All`), plus the capstone write-up in Module 10.


## 0.1 · What Lecture 3 actually covers — and what this lab does with it

This lab was designed by inspecting the Lecture 03 notebook (`Lecture 03_Trading Strategies _ Paradigms.ipynb`) cell by cell. The map below is your guarantee that the lab is *the lecture, made practical* — nothing imported from an unrelated curriculum.

| Lecture 03 content (as taught) | Where you implement it | Status |
|---|---|---|
| Algo-trading system process: strategy → programming → software → data → infrastructure → regulation | §1 architecture; Module 0 data; Capstone workflow | Required (conceptual) |
| Trend following; moving averages; breakout; channel trading | Modules 1, 3 (SMA, dual-SMA, Donchian) | Required |
| Momentum: RSI, MACD, ROC, time-series momentum; stochastic & HF momentum | Module 1 (ROC), Module 4 (RSI); MACD as EMA-spread reading | Required: RSI/ROC · MACD: Stretch |
| Showcase 1: price vs SMA-100 crossover on **GS**, `Signal ∈ {+1,-1}`, trades where `Signal.diff() == ±2` | Module 2 (audited, long/flat vs long/short) | Required |
| "Naive backtest": sweep SMA 2–100, rank by cumulative return, explicit warning re costs/slippage/sample sensitivity | Module 5 (costs added), Module 8 (sweep done *properly*: train/test) | Required — and critiqued |
| Showcase 2: dual moving-average crossover (5/20 described; 50/200 run on **AAPL**) | Module 3 | Required |
| Technical analysis with `pandas-ta`: Bollinger Bands on **TSLA** | Module 1/4 — bands built **from scratch** in pandas; `pandas-ta` only as a cross-check (Stretch) | Required (from scratch) |
| RSI showcase on TSLA (interactive RSI; "sections" and a profit calculation) | Module 4 (RSI from scratch); Appendix B (audit of the lecture's profit logic) | Required |
| Mean reversion: types, tools (MA, Bollinger, RSI, z-score), risks & limitations | Module 4 | Required |
| Single-stock mean reversion: cumulative mean µ, $z=(x-\mu)/\sigma$, short if $z>1$, long if $z<-1$, flat if $|z|<0.5$ | Module 4 — with a required comparison of *cumulative* vs *rolling* statistics | Required |
| Pairs trading: spread, cointegration (`statsmodels.coint`), **KO/PEP**, screener over S&P names, OLS hedge ratio, spread z-score | Module 9 | **Optional extension** (two-leg accounting is beyond the core workflow) |
| Lecture 01: acquiring/cleaning data via APIs and CSV; Lecture 02: multi-stock panels, trend-template screening | Module 0; Module 8 transfer test; Module 9 screener logic | Required |
| Backtrader, ML, regression/factor models, live APIs | — | **Out of scope** (Lectures 04–08). Not needed, not graded. |

### Learning objectives
By the end you can, without following a recipe:
1. Download, validate, and document real daily OHLCV data — and say exactly what you did *not* do to it.
2. Build SMA, EMA, Donchian channels, Bollinger/z-scores, RSI, and ROC from first principles in pandas/NumPy.
3. Turn a plain-English hypothesis into explicit entry/exit/holding rules, and separate **signal → position → trade**.
4. Backtest both vectorized and with an iterative trade simulator, stating execution assumptions and charging costs.
5. Detect look-ahead bias and audit someone else's backtest code (including Lecture 03's own shortcuts).
6. Evaluate strategies with CAGR, volatility, Sharpe, maximum drawdown, trades/win-rate, exposure — against buy-and-hold, under identical assumptions.
7. Run robustness checks (train/test, sub-periods, cost and delay sensitivity, a second instrument) and write an honest verdict.


## 0.2 · Assignment specification & grading

| Module | What you produce | Pts |
|---|---|---:|
| M0 Real data | Validated SPY dataset + data-quality report | 10 |
| M1 Indicators | Six indicators from scratch, verified & plotted | 12 |
| M2 Signal architecture | Signal/position/trades + look-ahead bug hunt with numbers | 12 |
| M3 Trend strategies | SMA, dual-SMA, Donchian rule tables + positions | 12 |
| M4 Reversion & momentum | Z-score/Bollinger and RSI strategies | 12 |
| M5 Vectorized backtester | Reusable net-of-cost engine passing property checks | 12 |
| M6 Event simulator | Trade log (entries, exits, reasons), vectorized-vs-loop comparison | 10 |
| M7 Performance | Metrics table vs benchmark + interpretation | 8 |
| M8 Robustness | Train/test sweep, sub-periods, cost/delay, transfer to AAPL | 8 |
| M9 Pairs (optional) | KO/PEP cointegration & spread strategy | +8 bonus |
| M10 Capstone | Own hypothesis → code → evaluation → 300–500-word verdict | 14 |
| **Total** | | **110 → scaled to 100 (M9 is bonus)** |

**Acceptance criteria for the whole lab:** `Restart & Run All` completes with no errors · every `CHECK` prints its ✅ line · no performance figure is typed by hand (all come from code) · the notebook states data source, tickers, date range, cost and execution assumptions · no synthetic prices anywhere.

### Setup
```bash
pip install -r requirements.txt   # pandas, numpy, matplotlib, yfinance, jupyter
# optional, only for Module 9: pip install statsmodels
```
Python ≥ 3.10. `pandas-ta` and `plotly` are **not** required — Lecture 03 uses them for display, but you will build the indicators yourself so you understand them; you may cross-check against `pandas-ta` as a Stretch task.


# 1 · The 80/20 Mental Model of Lecture 3

Five ideas carry almost all the practical value in Lecture 03. Everything else in this lab is an application of one of them.

| # | Core concept | One-line mental model | Why it pays |
|---|---|---|---|
| 1 | **Paradigm = a bet on persistence vs reversion** | Trend/momentum bets "what moved will keep moving"; mean reversion bets "what stretched will snap back". Neither is true in general — each is true in some regimes. | Stops you asking "which indicator is best?" and starts you asking "what behaviour am I betting on, and when does it fail?" |
| 2 | **Signal ≠ Position ≠ Trade** | A *signal* is an opinion formed at a close; a *position* is what you actually hold during the next bar; a *trade* is a *change* in position. | This one distinction is the difference between a backtest and a fiction (look-ahead bias). |
| 3 | **An indicator is compressed history, not a forecast** | SMA/EMA, Donchian, Bollinger/z, RSI, ROC are deterministic summaries of past prices. They contain no information that isn't in the price series. | Kills indicator-shopping. Value comes from the *rule + risk + cost* system around the summary. |
| 4 | **A backtest is accounting, not simulation magic** | $\text{NetRet}_t = \text{Position}_t \times \text{AssetRet}_t - \text{cost}\times|\Delta\text{Position}_t|$, compounded. If you can't trace one day's return by hand, you don't understand your result. | Makes every result debuggable and every "bug vs genuine improvement" question answerable. |
| 5 | **Evaluation = risk-adjusted, benchmarked, out-of-sample** | Total return alone rewards hidden risk and hidden exposure. CAGR/vol/Sharpe/drawdown vs buy-and-hold, on data the parameters weren't chosen on, is the minimum honest report. | Lecture 03's own SMA 2–100 sweep shows the trap: the "best" historical window is mostly luck (§M8 proves it on real data). |

Supporting ideas you will use repeatedly: **stateful holding** (enter on one condition, exit on another, hold in between) · **warm-up** (a 200-day SMA doesn't exist for 199 days — that is missing *information*, not missing *data* to fill) · **correlation ≠ cointegration** (Module 9) · **regime dependence** (a strategy is a claim about a regime; §M8 tests it across real regimes: 2015–19, 2020 crash/rebound, 2022 bear, 2023–24 bull).

## 1.1 · The system you are building

```
                 ┌──────────────────────── feedback: robustness checks (§M8) ───────────────────────┐
                 │                                                                                  │
 HYPOTHESIS → DATA → INDICATORS → RULES/SIGNAL → POSITION (lagged) → EXECUTION ASSUMPTIONS → RETURNS (net) → EVALUATION
 "price above    real     SMA/EMA,     +1/0/-1      Signal.shift(1)   next-bar fill,          Position×Ret   CAGR, Sharpe,
  its 200-day    OHLCV    Donchian,    stateful     held overnight   costs, slippage,        − fees         max DD, trades,
  average tends  (§M0)    z, RSI, ROC  (ffill)      (§M2)            overnight (§M5/M6)      (§M5)          vs benchmark (§M7)
  to keep rising" (§M1)    (§M3/M4)                                                              │
                                                                                    CAPSTONE (§M10) repeats the loop alone
```
Separation matters: **data preparation, strategy logic, execution assumptions, and evaluation live in different cells/functions** so you can change one without silently changing the others.

## 1.2 · Plain English → Python (the patterns you will reuse all lab)

| Plain English | pandas pattern | Watch out |
|---|---|---|
| "Average of the last 50 closes, today included" | `s.rolling(50).mean()` | First 49 values are `NaN` — correct, not an error |
| "Recent days count more" | `s.ewm(span=20, adjust=False).mean()` | `span` ≠ window length; Lecture 03's RSI uses `alpha=1/14` form |
| "Highest high of the *previous* 20 days" | `s.shift(1).rolling(20).max()` | Without `.shift(1)` today's bar is inside its own channel — it can never break out |
| "Yesterday's value" | `s.shift(1)` | The execution lag. `shift(-1)` is look-ahead — never use it on signals |
| "Day's % change" | `s.pct_change()` | First value `NaN`; fill with 0 only in *return accounting*, never in prices |
| "Hold until told otherwise" | `pd.Series(raw).ffill().fillna(0)` | Turns entry/exit *events* into a held *state* |
| "Long when condition, else flat" | `np.where(cond, 1, 0)` | Decide: long/flat `{0,1}` or long/short `{-1,+1}`? Different exposure, different risk — §M2 |
| "How many σ from its recent mean" | `(s - s.rolling(20).mean()) / s.rolling(20).std()` | Full-sample `.mean()/.std()` leaks the future (Appendix B) |

## 1.3 · Misconceptions this lab is designed to break
1. *"The signal uses today's close, so I earn today's return."* No — you earn the return **after** the position is on (§M2 quantifies the inflation).
2. *"Best parameter on the whole sample = best strategy."* That's selection on the answer (§M8).
3. *"RSI < 30 means price must bounce."* RSI is a bounded transform of recent gains/losses; in a downtrend it can stay low while price keeps falling.
4. *"Mean reversion with the mean of **all data so far / all data ever** is the same as a rolling mean."* Cumulative statistics go stale; full-sample statistics leak (§M4, Appendix B).
5. *"Fill missing prices forward so the series is complete."* That fabricates bars. Non-trading days aren't missing; genuinely missing rows get reported and dropped — never invented (§M0).
6. *"High win rate = good strategy."* Trend systems often win < 50% and profit from asymmetric payoffs. Win rate is meaningless without average win/loss (§M7).
7. *"The strategy beat buy-and-hold, so it's better."* Not if it held far more risk, or was simply long 95% of the time with extra fees. Compare risk *and exposure* (§M7).


# Module 0 · Real Historical Data — Acquire, Inspect, Validate

**Objective.** Produce one clean, documented daily dataset that every later module shares. **Relevance:** every professional failure mode in this course (biased backtests, phantom returns) starts as a data decision nobody wrote down.

**Data plan (fixed for the whole lab — do not silently change it)**

| Item | Choice | Why |
|---|---|---|
| Primary instrument | **SPY** (SPDR S&P 500 ETF) | Lecture 03's strategies are single-asset timing rules; SPY is highly liquid, has decades of Yahoo history, and *is* the benchmark — so "timing SPY vs holding SPY" is a clean question with no stock-picking confound |
| Transfer instrument | **AAPL** (used in Lectures 02–03) | Same rules, unchanged parameters, different animal: single-stock volatility, earnings jumps, a 4:1 split (Aug 2020). Tests whether a rule was about *behaviour* or about *SPY* |
| Pairs extension | **KO / PEP** | The exact pair Lecture 03 uses for cointegration (substitute products, same industry) |
| Source | Yahoo Finance via `yfinance` | Lecture 01/03's source; free; no API key |
| Range | `2015-01-01` → `2025-01-01` (i.e. through 2024-12-31) | 10 years ≈ 2,500 trading days: contains calm bull (2015–19), COVID crash & rebound (2020), bear market (2022), AI-led bull (2023–24) — real regimes, not engineered ones. Fast to download, kind to a laptop |
| Timeframe / columns | Daily `Open, High, Low, Close, Volume` (+ `Adj Close` retained) | Matches Lecture 03's OHLC usage |
| Price convention | Indicators & returns on `Close`, with `auto_adjust=False` (as in Lecture 03) | Yahoo split-adjusts historical OHLC, so the AAPL split won't fake a −75% crash; dividends are *not* in `Close`, so buy-and-hold slightly **understates** total return — you will state this limitation and test `Adj Close` as a Stretch |

**Concepts tested:** API download, MultiIndex column handling, index hygiene, missing/duplicate/invalid-value policy, warm-up vs missing data.

### Tasks
- **0.1** Download SPY with the plan above. If `yfinance` returns MultiIndex columns (recent versions do), flatten them. Sort by date; keep the columns listed.
- **0.2** Validate: monotonic unique DatetimeIndex · no duplicate dates · `Close > 0`, `Volume >= 0` · `High >= max(Open, Close)` and `Low <= min(Open, Close)` (allow tiny rounding tolerance) · count missing values per column.
- **0.3** Apply the lab's missing-data policy: **report, then drop rows with missing/invalid OHLC; never forward-fill or invent a price.** (Lecture 03's RSI demo reindexes to calendar days and forward-fills — that manufactures weekend prices. We do the opposite here.) Keep a `data_quality` summary (rows before/after, drops with reasons).
- **0.4** Cache the cleaned frame to `data/spy_daily.csv` and prove a reload round-trips. If the download fails (no network, Yahoo outage): retry later, or load a *real* CSV export (e.g. from your broker/Nasdaq.com) with the identical columns and record its source in the reflection — synthetic substitutes are not acceptable.
- **0.5** Plot `Close` (full range) and a 2019–2021 zoom. Annotate, in your reflection, where the COVID crash and 2022 bear market are.

**Expected outputs:** `prices` DataFrame (≈2,400–2,600 rows — provider revisions make the exact count vary; your CHECK asserts a range, not a magic number) · printed data-quality report · two plots.
**Pitfalls:** trusting `df.head()` as validation · `pct_change` on an unsorted index · treating warm-up `NaN`s from indicators (Module 1) as data errors · timezone-aware index surprises.


In [ ]:
# SETUP — provided. Run once; every module uses these names.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.width", 120)

START, END = "2015-01-01", "2025-01-01"   # END is exclusive in yfinance -> last bar is 2024-12-31
PRIMARY, TRANSFER = "SPY", "AAPL"
PAIR = ["KO", "PEP"]
TRADING_DAYS = 252
REQUIRED_COLS = ["Open", "High", "Low", "Close", "Volume"]

# pip install yfinance  (if needed)
import yfinance as yf
print("Setup OK")


In [ ]:
# TODO 0.1-0.3: download, flatten, validate, clean SPY.
# Hints: yf.download(PRIMARY, start=START, end=END, auto_adjust=False, progress=False)
#        - recent yfinance returns MultiIndex columns even for one ticker: handle both cases
#        - build a `data_quality` dict: rows_raw, duplicate_dates, missing_per_column, invalid_ohlc_rows, rows_clean
# Policy: report problems, drop rows with missing/invalid OHLC, NEVER fill a price.
raw = ...            # download result
prices = ...         # cleaned DataFrame indexed by date, REQUIRED_COLS present, sorted ascending
data_quality = ...   # dict as described
print(data_quality)
prices.head()


In [ ]:
# CHECK 0 — do not edit
assert isinstance(prices.index, pd.DatetimeIndex), "Index must be dates"
assert prices.index.is_monotonic_increasing and prices.index.is_unique, "Index must be sorted & unique"
assert all(c in prices.columns for c in REQUIRED_COLS), "Missing required OHLCV columns"
assert 2000 < len(prices) < 3000, f"Unexpected row count {len(prices)} — check ticker/range"
assert (prices["Close"] > 0).all() and (prices["Volume"] >= 0).all()
assert not prices[REQUIRED_COLS].isna().any().any(), "Cleaned data must have no NaN in OHLCV"
tol = 1e-6
assert (prices["High"] + tol >= prices[["Open", "Close"]].max(axis=1)).all(), "High/Low inconsistency"
assert (prices["Low"] - tol <= prices[["Open", "Close"]].min(axis=1)).all(), "High/Low inconsistency"
print(f"✅ CHECK 0 passed — {len(prices)} real trading days, {prices.index[0].date()} to {prices.index[-1].date()}")


In [ ]:
# TODO 0.4-0.5: cache to data/spy_daily.csv, reload, assert round-trip equality of Close;
# then plot full-range Close and a 2019-2021 zoom (two separate figures, labelled axes).
...


> **REFLECT 0 (write below):** (a) Your exact data source, download date, rows before/after cleaning and what you dropped. (b) Why is forward-filling a missing close different from a non-trading day? (c) One thing daily OHLCV *cannot* tell you about what happened inside a day.
>
> **Your answer:** …


# Module 1 · Indicators from First Principles

**Objective.** Build every indicator Lecture 03 uses — yourself, in pandas. **Relevance:** an indicator you cannot reproduce by hand is a black box you cannot debug, and "the library said so" is not a research method.

**Reference (formulas only — implementation is yours)**
- $\text{SMA}_N$: mean of last $N$ closes · $\text{EMA}$: `ewm(span=N, adjust=False)`, recent days weighted more
- Donchian (prior): $\max/\min$ of the **previous** $N$ closes · Bollinger: $\text{SMA}_{20} \pm 2\sigma_{20}$ · z-score: $(P-\mu_{20})/\sigma_{20}$
- RSI(14): split daily changes into gains/losses, smooth each with $\alpha=1/14$, $RS=\bar G/\bar L$, $RSI=100-100/(1+RS)$
- ROC(12): $\%\,$change vs 12 days ago (Lecture 03's rate-of-change momentum)

### Tasks (on `prices`, in a working copy `ind`)
1. `SMA_50`, `SMA_200`, `EMA_12`, `EMA_26`, and `EMA_SPREAD = (EMA_12-EMA_26)/EMA_26`.
2. `DC_HIGH_20`, `DC_LOW_20` — **prior** 20-day channel (shifted). `ROLL_MEAN_20`, `ROLL_STD_20`, `Z_20`, Bollinger `BB_UP`, `BB_LO`.
3. `RSI_14` from scratch (no `pandas_ta`). Handle the all-gains edge case so RSI stays in $[0,100]$.
4. `ROC_12`.
5. Verify **one** SMA value and **one** RSI value by an independent recomputation from raw closes (a few lines, different method — e.g. explicit slice mean for the SMA; a hand loop over ≤30 rows for RSI is acceptable here precisely because it's a *check*, not the strategy).
6. Plot price + SMA_50/SMA_200 (2018–2020 window) and a second figure: RSI with 30/70 lines.

**Expected outputs:** `ind` with all columns · verification printout agreeing to ~1e-8 (SMA) / ~0.1 (RSI, smoothing conventions differ slightly) · two plots.
**Pitfalls:** Donchian without the shift · using full-sample mean/std for z · RSI division-by-zero · dropping warm-up rows from `prices` itself (keep them; indicators are `NaN`, prices are real) · `ewm(span=...)` vs `ewm(alpha=...)` confusion.
**STRETCH:** install `pandas-ta`, compare its RSI/Bollinger to yours, and explain any discrepancy (Wilder smoothing conventions) in two sentences.


In [ ]:
# TODO 1.1-1.4: build all indicators on a copy of prices.
ind = prices.copy()
# SMA_50, SMA_200, EMA_12, EMA_26, EMA_SPREAD ...
# DC_HIGH_20, DC_LOW_20 (PRIOR channel) ...
# ROLL_MEAN_20, ROLL_STD_20, Z_20, BB_UP, BB_LO ...
# RSI_14 from scratch: diff -> gain/loss -> ewm(alpha=1/14) -> RS -> RSI ...
# ROC_12 ...
ind[["Close","SMA_50","SMA_200","Z_20","RSI_14","ROC_12"]].tail()


In [ ]:
# CHECK 1 — do not edit (assumes the column names in the task)
need = ["SMA_50","SMA_200","EMA_12","EMA_26","EMA_SPREAD","DC_HIGH_20","DC_LOW_20",
        "ROLL_MEAN_20","ROLL_STD_20","Z_20","BB_UP","BB_LO","RSI_14","ROC_12"]
assert all(c in ind.columns for c in need), [c for c in need if c not in ind.columns]
assert ind["SMA_200"].iloc[:199].isna().all() and ind["SMA_200"].iloc[199:].notna().all(), "SMA warm-up wrong"
rsi_ok = ind["RSI_14"].dropna()
assert ((rsi_ok >= 0) & (rsi_ok <= 100)).all(), "RSI out of [0,100]"
# Prior-channel property: today's close must be ABLE to exceed DC_HIGH_20 (impossible if unshifted)
assert (ind["Close"] > ind["DC_HIGH_20"]).sum() > 0, "Donchian looks unshifted — today's bar is inside its own channel"
bb = ind.dropna(subset=["BB_UP", "BB_LO", "ROLL_MEAN_20"])
assert (bb["BB_UP"] >= bb["ROLL_MEAN_20"]).all() and (bb["BB_LO"] <= bb["ROLL_MEAN_20"]).all()
print("✅ CHECK 1 passed — indicators built; warm-up, bounds and prior-channel properties hold")


In [ ]:
# TODO 1.5-1.6: independent verification of one SMA value and one RSI value + the two required plots.
...


> **REFLECT 1:** Pick one indicator. In ≤3 sentences: what past-price information does it keep, what does it throw away, and what trading behaviour would that make it blind to?
>
> **Your answer:** …


# Module 2 · Signal → Position → Trade, and the Look-Ahead Bug Hunt

**Objective.** Implement Lecture 03's Showcase-1 architecture correctly, then *prove* — with numbers on real data — why the lag is non-negotiable. **Relevance:** this is the single highest-value skill in the lab; most fictional backtests die here.

**The contract (memorise it)**
- `Signal_t ∈ {0,1}` (long/flat) or `{-1,+1}` (long/short): opinion **after** close $t$. Held state, not an event.
- `Position_t = Signal_{t-1}` → `Position = Signal.shift(1).fillna(0)`. You hold during day $t$ what you decided at close $t-1$.
- A **trade** is `Position.diff() != 0`. (Lecture 03's `Signal.diff() == ±2` finds the same events for the `{-1,+1}` encoding — you will verify that equivalence, and see why `diff()` on `{0,1}` differs.)

### Tasks
1. **Lecture reproduction, corrected:** `SIG_SMA100`: 1 when `Close > SMA_100` else 0 (long/flat), then `POS_SMA100` lagged. Also build Lecture 03's original `{-1,+1}` version and its trade markers; confirm both encodings agree on *entry dates* (they will not agree on exposure — note why).
2. **Trade table:** from `POS_SMA100`, extract entries/exits (date, price, side). Count trades; compute mean holding length in bars. Positions are held **overnight** — state that explicitly.
3. **Bug hunt.** Three fragments below each contain one look-ahead error. For each: name the error, fix it, and quantify the damage on SPY by running both versions through *gross* returns (`Position × pct_change`, no costs yet — engine arrives in M5): report final wealth buggy vs correct.
   - **A:** `ret = df['Signal'] * df['Close'].pct_change()` (no lag)
   - **B:** `z = (df['Close'] - df['Close'].mean()) / df['Close'].std()` (whole-sample statistics)
   - **C:** Donchian channel from `rolling(20).max()` with no shift, traded same-day.
4. Plot `Close` + `SMA_100` with entry/exit markers for 2019–2020 only (full-range marker plots are unreadable — say so if yours is).

**Expected outputs:** positions with lag invariant checked · trade table · a 3-row bug table (bug, fix, wealth buggy vs correct) · one readable plot.
**Pitfalls:** shifting the *returns* instead of the signal · `fillna(0)` before vs after shifting · counting every day with `Position==1` as a "trade".


In [ ]:
# TODO 2.1-2.2: SMA-100 signal (long/flat AND lecture's -1/+1), lagged positions, trade table.
sig = ind.copy()   # reuse Module 1 frame; add SMA_100 if needed
...
trades_sma100 = ...   # DataFrame: entry_date, entry_price, exit_date, exit_price (exit may be open)
print(f"Trades: {len(trades_sma100)}")
trades_sma100.head(10)


In [ ]:
# CHECK 2 — do not edit (column names from the task: SIG_SMA100, POS_SMA100 on `sig`)
assert (sig["POS_SMA100"].iloc[1:] == sig["SIG_SMA100"].iloc[:-1].values).all(), "Position must lag Signal by exactly 1 bar"
assert sig["POS_SMA100"].iloc[0] == 0
assert set(sig["POS_SMA100"].dropna().unique()) <= {0, 1}
print("✅ CHECK 2 passed — 1-bar execution lag verified")


In [ ]:
# TODO 2.3: bug hunt. For bugs A, B, C build buggy vs corrected gross wealth on SPY and tabulate.
# Keep it gross (no costs) — you are isolating the bias, not the strategy.
bug_table = ...   # DataFrame with columns: bug, error, wealth_buggy, wealth_correct
bug_table


> **REFLECT 2:** Bug A usually *helps* reported performance. Explain the mechanism in ≤4 sentences (what information is used before it exists, and on which days the fake profit concentrates — check days when the signal flips).
>
> **Your answer:** …


# Module 3 · Trend Strategies (Lecture 03, Showcases 1–2 + Breakout)

**Objective.** Implement the lecture's three trend rules as **explicit, stateful rule tables** — before any returns are computed. **Relevance:** if you cannot write the rules without code, you don't have a strategy, you have a hope.

| Strategy | Entry (→ Signal 1) | Exit (→ Signal 0) | Hold | Encoding |
|---|---|---|---|---|
| T1 Price/SMA | `Close > SMA_100` | `Close <= SMA_100` | — (stateless) | long/flat |
| T2 Dual-SMA | `SMA_50 >= SMA_200` | `SMA_50 < SMA_200` | — (stateless) | long/flat |
| T3 Donchian-20 | `Close > DC_HIGH_20` (prior) | `Close < DC_LOW_20` (prior) | **stateful (`ffill`)** | long/flat |

### Tasks
1. Implement T1–T3 signals + lagged positions (`SIG_/POS_` prefixed columns on separate copies — never overwrite Module 1's `ind`).
2. For each: exposure (% days long), number of trades, mean holding length. Tabulate. *(No returns yet — that separation is deliberate.)*
3. Plot T3's channel + position shading for a 2-year window; plot T2's two SMAs for the same window.
4. **Whipsaw investigation:** count T2 trades inside calendar 2018 (choppy) vs 2017 (trending). Connect the counts to Lecture 03's whipsaw discussion — using *your* counts, not the lecture's prose.
5. STRETCH (buffer): re-run T2 requiring `EMA_SPREAD > +0.5%` to enter and `< −0.5%` to exit (stateful). Report how trade count changes; returns come in M7.

**Expected outputs:** three position series passing the lag CHECK pattern · exposure/trades/holding table · two plots · whipsaw paragraph.
**Pitfalls:** stateless logic for T3 (it *must* remember) · comparing `Close` to an unshifted channel · letting `SMA_200` warm-up days become accidental long positions.


In [ ]:
# TODO 3.1-3.2: T1, T2, T3 signals/positions + exposure & trade-count table (no returns yet).
# REQUIRED CONVENTION: trend_frames has keys 'T1_SMA100', 'T2_DUALSMA', 'T3_DONCHIAN';
# each value is a DataFrame with at least columns ['Close', 'Signal', 'Position'] using THAT strategy's lagged Position.
trend_frames = {}
trend_stats = ...   # DataFrame: strategy, exposure_pct, n_trades, mean_holding_bars
trend_stats


In [ ]:
# TODO 3.3-3.4: Donchian + dual-SMA plots (2-year window) ; whipsaw counts 2017 vs 2018 for T2.
...


In [ ]:
# CHECK 3 — do not edit
assert set(trend_frames) == {"T1_SMA100", "T2_DUALSMA", "T3_DONCHIAN"}, trend_frames.keys()
for name, f in trend_frames.items():
    assert {"Close", "Signal", "Position"} <= set(f.columns), (name, f.columns)
    assert set(f["Position"].dropna().unique()) <= {0, 1}, name
    assert (f["Position"].iloc[1:] == f["Signal"].iloc[:-1].values).all(), f"{name}: Position must lag Signal by 1"
    assert 0 < f["Position"].mean() < 1, f"{name}: exposure should be strictly between 0 and 1 on 2015-2024 SPY"
print("✅ CHECK 3 passed — all three trend strategies are lagged, long/flat, and sometimes in/out")


# Module 4 · Mean Reversion & Momentum Oscillators (Lecture 03, second half)

**Objective.** Build the lecture's two counter-trend ideas with the same rigour as the trend rules — and settle, empirically, a question Lecture 03 leaves open: *cumulative* vs *rolling* statistics.

| Strategy | Entry (→ 1) | Exit (→ 0) | Notes |
|---|---|---|---|
| R1 Z-score | `Z_20 <= -1.5` | `Z_20 >= 0` (back to mean) | Asymmetric: enter far, exit at the mean — not at the opposite extreme |
| R2 Bollinger | `Close <= BB_LO` | `Close >= ROLL_MEAN_20` | **First prove** R2's entry is (almost) R1 with a different threshold — find the z equivalent of the band |
| R3 RSI pullback | `RSI_14 < 30` | `RSI_14 > 65` | Stateful; Lecture 03's "oversold/overbought sections" made executable |

### Tasks
1. Implement R1 and R3 (signals, lagged positions, exposure/trade stats — same table format as M3 so M7 can merge them).
2. **Cumulative vs rolling (required experiment):** rebuild R1 using Lecture 03's single-stock recipe — cumulative expanding mean/std up to each day. Compare: number of entries per year (2016 vs 2023), and describe what happens to the z-scale as the sample grows. Conclude which you will carry forward and why.
3. Show, in one printed check, the z-threshold that makes Bollinger entry equivalent to R1-style entry; then implement R2 *or* justify dropping it as a duplicate (duplicated strategies in a comparison are a real research error — say which you did).
4. Plot 2019–2020: price with R1 entry/exit markers, and RSI in a panel below with R3 markers.
5. STRETCH: Lecture 03 also goes **short** at $z>1$. Add a long/short R1 variant; keep it clearly separated — M7 will show what shorting does to exposure and drawdown on an index that drifts upward.

**Expected outputs:** R1/R3 positions passing lag checks · cumulative-vs-rolling comparison (counts + paragraph) · equivalence check printout · one two-panel plot.
**Pitfalls:** exiting R1 at $z\ge+1.5$ "for symmetry" (changes the strategy's whole character — holding through the mean) · RSI signals that re-enter every day while RSI < 30 (stateful holding fixes this; verify your trade count is sane) · using adjusted/unadjusted prices inconsistently with M1.


In [ ]:
# TODO 4.1-4.3: R1 (rolling z), R3 (RSI), cumulative-z experiment, Bollinger equivalence check.
# REQUIRED CONVENTION: rev_frames has keys 'R1_ZSCORE', 'R3_RSI'; each has ['Close', 'Signal', 'Position'].
rev_frames = {}
rev_stats = ...
cumulative_experiment = ...   # per-year entry counts, rolling vs cumulative
print(cumulative_experiment)
rev_stats


In [ ]:
# CHECK 4 — do not edit
assert set(rev_frames) == {"R1_ZSCORE", "R3_RSI"}, rev_frames.keys()
for name, f in rev_frames.items():
    assert {"Close", "Signal", "Position"} <= set(f.columns), (name, f.columns)
    assert set(f["Position"].dropna().unique()) <= {0, 1}, name
    assert (f["Position"].iloc[1:] == f["Signal"].iloc[:-1].values).all(), f"{name}: Position must lag Signal by 1"
print("✅ CHECK 4 passed — reversion/momentum strategies are lagged and long/flat")


In [ ]:
# TODO 4.4: two-panel plot (price + R1 markers; RSI + R3 markers), 2019-2020.
...


> **REFLECT 4:** Lecture 03 lists "fundamental changes make the historical mean irrelevant" as a mean-reversion risk. Give one concrete SPY episode in 2015–2024 where buying a $z\le-1.5$ dip was buying a *regime change*, and say how your R1 rules behaved there (from your markers/trade table — cite dates).
>
> **Your answer:** …


# Module 5 · The Vectorized Backtester — Returns After Costs

**Objective.** One reusable engine that turns *any* position column into net returns, used identically by every strategy (Lecture 03's naive backtest, productionised). **Relevance:** comparability. If each strategy has its own return code, differences may be code, not strategy.

**Specification — `backtest_positions(df, position_col, fee_bps=10, slippage_bps=5)`**
- `AssetRet_t = Close.pct_change()` (first bar → 0 in accounting only)
- `GrossRet_t = Position_t × AssetRet_t` — `Position` must already be lagged (the engine **asserts** nothing about this; *you* guarantee it — see CHECK 2 pattern)
- `Turnover_t = |Position_t − Position_{t−1}|` (first bar: position itself) · `Cost_t = Turnover_t × (fee+slippage)/10⁴`
- `NetRet = GrossRet − Cost` · `Wealth = (1+NetRet).cumprod()` starting at 1.0
- Returns a frame with `AssetRet, GrossRet, Turnover, Cost, NetRet, Wealth`. Position held **overnight**, close-to-close — write that assumption in the docstring.

**Execution-assumption box (state it, don't hide it):** signal at close $t$ → position earns close-to-close return on $t{+}1$. This is the standard daily-bar approximation: it does **not** claim you filled at close $t$'s price the instant the signal printed. Module 6 prices fills at the **next open** instead, and you will reconcile the two.

### Tasks
1. Implement the engine to spec. No strategy logic inside it.
2. **Property tests (required, before any strategy):** (a) flat position ⇒ Wealth ≡ 1 · (b) buy-and-hold `Position≡1` ⇒ Wealth matches a direct `(1+AssetRet).cumprod()` minus exactly one entry cost · (c) raising costs never raises final Wealth (test 0 vs 25 bps on T2) · (d) hand-trace one day: pick a day, recompute `NetRet` from raw closes in two independent lines, assert agreement.
3. Run the engine (gross, i.e. 0 bps, and net 10+5 bps) on T1–T3, R1, R3. Report final Wealth gross vs net side by side — the gap *is* the cost of turnover.
4. Reproduce Lecture 03's naive result *correctly*: sweep single SMA windows on SPY gross, rank — save the ranking; M8 will interrogate it. (Do **not** conclude anything yet.)

**Pitfalls:** charging cost on `Signal` instead of `Position` changes · `cumprod` over `NaN` · percent vs decimal (bps!) · silently re-shifting inside the engine (double lag).


In [ ]:
def backtest_positions(df, position_col, fee_bps=10, slippage_bps=5):
    # TODO 5.1: implement to the Module 5 spec. Add a docstring stating the execution assumptions.
    ...

# TODO 5.2: property tests (a)-(d) as asserts/prints
...


In [ ]:
# CHECK 5 — do not edit (uses your engine; strategy frames from M3/M4 must expose Position columns)
# Uses the mandated M3/M4 conventions: trend_frames['T2_DUALSMA']['Position'], rev_frames['R1_ZSCORE']['Position'].
flat = prices.copy(); flat["Position"] = 0
w_flat = backtest_positions(flat, "Position")["Wealth"]
assert (w_flat == 1.0).all(), "Flat position must keep Wealth at 1.0"
bnh = prices.copy(); bnh["Position"] = 1
res_bnh = backtest_positions(bnh, "Position", fee_bps=10, slippage_bps=5)
direct = (1 + prices["Close"].pct_change().fillna(0)).cumprod()
expected_last = direct.iloc[-1] * (1 - 15/10000)   # exactly one entry cost at 15 bps total
assert abs(res_bnh["Wealth"].iloc[-1] - expected_last) / expected_last < 1e-9, "Buy-and-hold accounting mismatch"
print("✅ CHECK 5 passed — flat & buy-and-hold accounting verified against an independent calculation")


In [ ]:
# TODO 5.3-5.4: gross vs net final Wealth for T1,T2,T3,R1,R3 ; SMA-window gross sweep ranking (save for M8).
sweep_ranking = ...
sweep_ranking.head(10)


# Module 6 · Iterative Simulation & the Trade Log (Vectorized vs Stateful)

**Objective.** Re-implement **one** strategy (T3 Donchian or R1) as an explicit bar-by-bar loop with a trade log — the thing vectorized code cannot give you. **Relevance:** stops, take-profits, max-holding and "exit because X happened first" are inherently sequential; Lecture 07's event frameworks are this loop, formalised.

**Specification**
- Iterate chronologically. State: `in_position`, entry date/price, bars held. You may not look at bar $t+1$ when deciding bar $t$.
- **Fills:** entry/exit at the **next bar's Open** after the signal bar (different assumption from M5 — deliberate).
- Exits, first one that fires: (i) strategy exit signal · (ii) stop-loss: close ≤ entry × (1 − 8%) · (iii) max holding 60 bars. Optional Stretch: take-profit +15%.
- Log per closed trade: entry/exit dates & prices, bars held, gross return, cost (15 bps × 2 sides), net return, exit reason. Close any open trade at the last bar and flag it `open_at_end`.
- **Daily-data limitation (required in your write-up):** with only OHLC you cannot know whether a stop and a profit target were *both* crossed intraday, or in which order. Your close-based stop avoids inventing that knowledge — state the trade-off.

### Tasks
1. Implement the loop for your chosen strategy. 2. Validate the log (no overlapping trades; exit ≥ entry date; net = gross − costs recomputed independently for 3 trades). 3. Compare loop vs M5 vectorized result for the same rules: final wealth, trade count — explain every source of difference (fill price, stop/max-hold, end handling) in ≤6 sentences. 4. Report the exit-reason breakdown.

**Expected outputs:** `trade_log` DataFrame · validation printout · comparison paragraph · exit-reason counts.
**Pitfalls:** peeking at the exit bar's close to decide the exit at its open · charging one side's cost · forgetting the still-open final trade (survivor of your own backtest!).


In [ ]:
# TODO 6.1-6.2: event loop for ONE strategy (next-Open fills, -8% close stop, 60-bar max hold).
trade_log = ...   # REQUIRED columns: entry_date, entry_price, exit_date, exit_price, bars_held,
                  # gross_return, net_return, exit_reason, open_at_end
print(trade_log.head()); print(trade_log["exit_reason"].value_counts())


In [ ]:
# CHECK 6 — do not edit
need = {"entry_date","entry_price","exit_date","exit_price","bars_held","gross_return","net_return","exit_reason","open_at_end"}
assert need <= set(trade_log.columns), need - set(trade_log.columns)
assert (trade_log["exit_date"] >= trade_log["entry_date"]).all()
assert (trade_log["bars_held"] >= 0).all() and (trade_log["entry_price"] > 0).all()
assert set(trade_log["exit_reason"]) <= {"strategy_exit", "stop_loss", "max_holding", "open_at_end", "take_profit"}
ordered = trade_log.sort_values("entry_date")
assert (ordered["entry_date"].iloc[1:].values >= ordered["exit_date"].iloc[:-1].values).all(), "Trades overlap — state machine bug"
print(f"✅ CHECK 6 passed — {len(trade_log)} non-overlapping trades with a complete audit log")


In [ ]:
# TODO 6.3: loop-vs-vectorized comparison (final wealth, n trades) + write the <=6-sentence explanation as a comment/print.
...


# Module 7 · Professional Performance Analysis

**Objective.** Evaluate everything — identically, against the benchmark — and learn to read the table like a risk manager, not a fan.

**Required metrics (each exists for a reason — state the reason in your interpretation)**
| Metric | Formula | Question it answers |
|---|---|---|
| Total return | $W_T-1$ | What did $1 become? (nothing about path or risk) |
| CAGR | $W_T^{252/N}-1$ | Comparable annual growth |
| Ann. volatility | $\text{std}(R_{net})\sqrt{252}$ | How bumpy was the ride? |
| Sharpe ($R_f=0$, stated) | CAGR / ann. vol* | Return per unit of bumpiness |
| Max drawdown & duration | $\min_t (W_t-\text{peak}_t)/\text{peak}_t$ | Worst peak-to-trough pain; how long underwater? |
| Exposure | mean(Position) | How much of the result is just "being long"? |
| Trades / win rate / avg trade | from M6 log (loop strategy) and position changes (others) | Behaviour & payoff shape — win rate never alone |

*Lecture 07 will refine Sharpe (excess returns, downside deviation/Sortino); the CAGR/vol form above is this course stage's convention — note its approximation.

### Tasks
1. Implement `performance_summary(wealth, net_ret, position, name)` returning the table row (trades/win-rate from a trade-return series where available; else `n/a` with a footnote — never fabricate).
2. One comparison table: Buy-and-Hold, T1, T2, T3, R1, R3 — **same dates, same 10+5 bps**, gross columns excluded. Plus equity-curve plot (all strategies + benchmark) and a drawdown plot (each strategy's underwater curve).
3. Interpretation (graded, REFLECT 7): rank by Sharpe and by total return — where do rankings disagree and why? Which strategy's return is most explained by exposure? Which drawdown would have been hardest to *sit through*, considering duration as well as depth?
4. Sanity: any strategy whose net Sharpe looks implausibly high (>2.5 on daily index timing) triggers a mandatory re-audit of M2/M5 before proceeding — write what you checked.

**Pitfalls:** annualising with calendar days · Sharpe on gross returns in a net table · comparing strategies over different warm-up windows (state your common evaluation window) · quoting win rate without average win/loss.


In [ ]:
def performance_summary(wealth, net_ret, position, name, trade_returns=None):
    # TODO 7.1: implement the Module 7 metric set. trade_returns: optional per-trade net returns.
    ...

# TODO 7.2: build the full comparison table (BnH, T1, T2, T3, R1, R3; net 10+5 bps) + equity & drawdown plots.
# REQUIRED CONVENTION: `comparison` is indexed by strategy name and has columns:
# ['Total Return (%)','CAGR (%)','Ann. Volatility (%)','Sharpe Ratio','Max Drawdown (%)','Exposure (%)','Trades']
comparison = ...
comparison


In [ ]:
# CHECK 7 — do not edit
need_cols = {"Total Return (%)","CAGR (%)","Ann. Volatility (%)","Sharpe Ratio","Max Drawdown (%)","Exposure (%)","Trades"}
assert need_cols <= set(comparison.columns), need_cols - set(comparison.columns)
assert len(comparison) >= 6, "Need benchmark + T1,T2,T3,R1,R3"
assert (comparison["Max Drawdown (%)"] <= 0).all()
assert comparison["Sharpe Ratio"].notna().all() and ((comparison["Exposure (%)"] >= 0) & (comparison["Exposure (%)"] <= 100)).all()
print("✅ CHECK 7 passed — comparison table is complete and internally consistent; now interpret it (REFLECT 7)")


> **REFLECT 7 (graded):** Answer the three interpretation questions in Task 3 here, citing numbers from *your* table. End with one sentence: "If forced to allocate to one of these tomorrow, I would choose ___ because ___" — risk reasoning required, return-only reasoning fails this reflection.
>
> **Your answer:** …


# Module 8 · Strategy Comparison & Robustness — Where Naive Backtests Die

**Objective.** Take Lecture 03's naive SMA sweep seriously enough to refute it properly, on real data. **Relevance:** this module is the difference between *finding* a strategy and *fooling yourself with in-sample selection*.

### Required checks (all on SPY unless stated; consistent 10+5 bps costs)
1. **Train/test sweep.** Split at 2020-01-01. Sweep single-SMA windows {10,20,50,100,150,200} (a deliberate subset of the lecture's 2–100). Rank on **train (2015–2019) only**, then report the chosen window's **test (2020–2024)** rank and metrics. Also report the test-best window. If they differ — they usually do — quantify what "pick the historical winner" would have cost.
2. **Sub-period regimes.** For T2, T3, R1, R3: CAGR & max DD in 2015–19 / 2020–21 / 2022 / 2023–24. Which paradigm owned which real regime? (This replaces the old simulated-regime idea with observed history.)
3. **Cost sensitivity.** Final wealth & Sharpe at total cost 0 / 15 / 30 / 50 bps for your highest-turnover and lowest-turnover strategies. At what cost does each cross below buy-and-hold?
4. **Delay sensitivity.** Add one extra lag day to T2 (`shift(2)` total). A real edge degrades gracefully; an artefact collapses. Report the change.
5. **Transfer test.** Apply T2's *unchanged* rules/parameters to AAPL (download & validate it exactly as in M0 — including noticing what the Aug-2020 split does to raw vs Yahoo-adjusted prices). Compare metrics; no re-tuning allowed. One paragraph: what transferred, what didn't, and why that was predictable from the instruments' characters.

**Expected outputs:** train/test table · sub-period table · cost & delay tables · AAPL comparison + paragraph.
**Pitfalls:** re-ranking on test ("it's just checking") · different evaluation windows per strategy · concluding from the transfer test that a rule is "good/bad" universally — one other asset is an anecdote with numbers, treat it as such.


In [ ]:
# TODO 8.1: train/test SMA sweep (rank on train, evaluate on test). No peeking: build the ranking from train only.
train_test = ...
train_test


In [ ]:
# TODO 8.2-8.5: sub-period table; cost sensitivity; extra-delay T2; AAPL transfer (validate like M0 first).
...


> **REFLECT 8:** In ≤5 sentences: what did the train/test exercise do to your confidence in Lecture 03's "rank all windows, pick the winner" procedure? What will you do differently in the capstone because of it?
>
> **Your answer:** …


# Module 9 · STRETCH — Pairs Trading (KO/PEP), Done Without Leaking

**Optional (+bonus). Prerequisites: Modules 0–7.** Lecture 03's finale, with its two silent biases fixed.

**Concepts:** correlation (co-movement of returns) ≠ cointegration (a *stationary spread*) · Engle–Granger `coint` (H₀: no cointegration; small p ⇒ evidence *for* a cointegrating relation) · hedge ratio from OLS of PEP on KO · spread $= P_{PEP} - b\cdot P_{KO}$ · trade the spread's z-score.

### Tasks
1. Download KO & PEP (M0 validation, shared trading days only — inner join, report dropped dates).
2. On **2015–2019 only**: correlation of daily returns, `coint` on prices, OLS hedge ratio. Report all three; explain in 2 sentences why correlation alone would not justify the trade.
3. Build the spread for 2015–2024 using the **train-estimated** hedge ratio. z-score with a **rolling** 60-day mean/std (Lecture 03's screener uses full-sample `zscore(spread)` — name that look-ahead explicitly in your notes).
4. Rules: long spread (long PEP / short KO leg notionals per hedge ratio) at $z\le-2$, short spread at $z\ge+2$, flat at $|z|\le0.5$, stateful, 1-bar lag. Backtest in *spread-return* terms with costs on both legs (state the simplification: you are modelling one unit of spread, not share-level accounting).
5. Report metrics (M7 function) for train vs test separately. Verdict in ≤4 sentences, including what would break this trade in real life (borrow, dividends on shorts, cointegration decay).

**Pitfalls:** estimating `b` or z on the full sample · `coint` on returns instead of prices · calling p < 0.05 "proof" — it is evidence against H₀, on *that* sample.


In [ ]:
# STRETCH TODO 9.1-9.4: KO/PEP download+validate, train-only coint/OLS, rolling-z spread strategy, train vs test metrics.
...


# Module 10 · Capstone — Your Hypothesis, Your Code, Your Verdict

**Translate one plain-English hypothesis into a tested strategy. No solution exists for this module — by design.**

Pick **one** (or propose your own and get it approved in one sentence at the top of your code):
- **A. Trend-filtered dip-buying:** "Oversold dips are worth buying only inside an uptrend." (e.g. trend filter from Lecture 03's SMAs + RSI trigger/exit — *you* choose windows, thresholds, and the exit hierarchy.)
- **B. Breakout with confirmation:** "A Donchian breakout that also satisfies a momentum condition (ROC or RSI — your choice) avoids false breakouts." Define "confirmation" precisely.
- **C. Regime switch:** "Trend rules in trending conditions, mean-reversion rules otherwise." You must define the regime *using only information available that day* (e.g. from Module 1 indicators) — the hard part is the switch, not the two halves.

### Required deliverables (all in this module)
1. **Hypothesis card** (before code): market behaviour bet on · why it might exist · regime where it should work · regime where it should fail · what result would make you *reject* it (written before you see results).
2. **Rule table:** indicators & parameters, entry, exit(s), position encoding, execution lag, costs (10+5 bps), instrument (SPY; AAPL transfer optional).
3. **Implementation** reusing your M5 engine (and M6 loop if you use stops/max-holding). New code only for indicators/rules.
4. **Bug audit:** deliberately run your M2 Bug-A variant on your capstone, show you can detect the inflation, then remove it.
5. **Evaluation:** M7 table row vs buy-and-hold & vs the closest single-paradigm strategy from M3/M4 · equity + drawdown plots · trade count & exposure.
6. **Two robustness checks** chosen from M8's menu (train/test, cost, delay, sub-period, transfer) — chosen *before* running them, stated in the hypothesis card addendum.
7. **Verdict (300–500 words):** reject / inconclusive / worth paper-trading (Lecture 07's forward test — name it as the next step, do not claim live readiness) · limitations: daily-bar execution, dividends/`Close` convention, single-asset & single-decade evidence, parameter choices · what you would test next.

**Acceptance:** hypothesis card predates results (write it first, don't edit it afterwards — add a dated addendum instead) · all numbers from code · at least one robustness check that *could have embarrassed* the strategy · verdict consistent with your own tables.


In [ ]:
# CAPSTONE — start with the hypothesis card as a comment/print BEFORE any strategy code.
# 10.1 Hypothesis card (fill every field):
hypothesis_card = {
    "hypothesis": "...",          # plain English, one sentence
    "why_it_might_exist": "...",
    "should_work_when": "...",
    "should_fail_when": "...",
    "i_will_reject_it_if": "...",
    "robustness_checks_chosen": ["...", "..."],
}
for k, v in hypothesis_card.items(): print(f"{k}: {v}")


In [ ]:
# TODO 10.2-10.5: rule table (as a printed DataFrame), indicators, signals, lagged positions,
# M5 backtest, M7 metrics vs benchmark & nearest M3/M4 strategy, bug-audit run, two robustness checks, plots.
...


> **CAPSTONE VERDICT (300–500 words, write here):** …
>
> **Limitations checklist to address explicitly:** execution on daily bars · `Close` vs total return · costs modelled vs real spreads · one asset / one decade · parameters chosen by you · what "one more year of data" could change.


# Appendix A · Reusable Reference — the practical 20%

**pandas survival kit** (verify each once; then trust but check): `rolling(w).mean()/std()/max()/min()` trailing window, `NaN` until warm · `ewm(span=n, adjust=False).mean()` EMA · `shift(1)` yesterday / lag a signal · `pct_change()` returns · `diff()` changes (position→trades) · `ffill()` carry a state forward · `cummax()` running peak (drawdowns) · `np.where(cond, a, b)` vectorised if/else · `.clip(lower=0)` keep positives (RSI gains).

**The five formulas:** $R_t=P_t/P_{t-1}-1$ · $\text{NetRet}_t=\text{Pos}_t R_t-\text{cost}\,|\Delta\text{Pos}_t|$ · $W_t=\prod(1+\text{NetRet})$ · $\text{CAGR}=W_T^{252/N}-1$ · $\text{DD}_t=(W_t-\text{cummax}(W)_t)/\text{cummax}(W)_t$.

**Verify-anything checklist:** ① recompute one indicator value from raw data by a second method ② assert `Position == Signal.shift(1)` ③ flat-position backtest ≡ 1.0 ④ buy-and-hold backtest vs direct cumprod ⑤ higher costs ⇒ lower wealth ⑥ count trades from `Position.diff()` and eyeball 5 of them on a plot ⑦ check exposure is plausible ⑧ re-run top-to-bottom in a fresh kernel.

**Reading a result table:** start at max drawdown & exposure, *then* Sharpe, *then* CAGR. Ask: "what was I paid for, and what pain bought it?"

# Appendix B · Audit Notes on Lecture 03's Own Code (read after M2/M4)
Lecture code is teaching code — fast, not production. Three shortcuts to be able to name: ① Showcase-1's `{-1,+1}` always-in encoding means the "strategy" is short half its life — compare like-for-like exposures before comparing returns. ② The RSI showcase's profit routine scores whether a *direction guess* matched the next move — that is not position P&L (no holding state, no compounding); your M5/M6 accounting replaces it. ③ The pairs screener's `zscore(spread)` uses the full-sample mean/std, and hedge ratios fit on the traded sample — both leak future information (fixed in M9 by train-only estimation + rolling z). Finding these is a skill, not a criticism: professional code review is exactly this.

# Appendix C · Limitations You Must Carry Forward
Daily OHLCV cannot establish intraday fill order (stop vs target), queue position, or true spreads · `Close` excludes dividends (total return understated; `Adj Close` is the remedy to test) · Yahoo data is revised and split/dividend-adjusted retroactively — record your download date · SPY timing results say nothing about stock selection, and 2015–2024 is one decade in one market · costs of 15 bps are a modelling choice — real spreads/commissions/borrow differ · survivorship: SPY's constituents changed over the decade; an ETF backtest inherits that silently.

# Submission checklist
☐ Restart & Run All passes · ☐ every CHECK ✅ · ☐ data source/range/costs/execution stated · ☐ no hand-typed results · ☐ no synthetic data · ☐ reflections answered from *your* tables · ☐ capstone hypothesis card written before results · ☐ verdict names limitations and a next test (paper trading, Lecture 07) — and promises no profit.
